In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="11-people-you-may-know",
    notebook_name="01_pymk_system_design.ipynb"
)

# People You May Know (PYMK) -- Full System Design

## The Big Idea (For a 12-Year-Old)

Imagine you just moved to a new school. You do not know anyone. But the school counselor is magic -- she already knows that your best friend from summer camp also knows three kids in your new class, that the kid next door goes to the same chess club as you, and that five people in your new school went to your old school.

She hands you a list: "Here are the kids you should talk to first." That is **People You May Know** (PYMK). LinkedIn and Facebook do exactly this, but for a billion people at once.

---

## Staff-Level Technical Summary

We design a **LinkedIn/Facebook-style PYMK system** using:
- **Friends-of-Friends (FoF)** as the candidate generation strategy (92% of new connections come from FoF)
- **Graph-based features**: common connections, Jaccard similarity, Adamic-Adar index
- **Edge prediction** formulation with GNN-based scoring
- **Pointwise vs pairwise Learning-to-Rank** comparison
- **Two-pipeline architecture**: offline batch generation + online serving
- **Offline evaluation**: ROC-AUC, mAP | **Online evaluation**: connection requests accepted

## 1. Problem Definition and Clarification

### The Interview Starts Here

In a real interview, you always start by asking clarifying questions. Here is the full set of questions and answers:

In [ ]:
# Let's organize the clarifying requirements as structured data
# This is how you'd think about it systematically in an interview

requirements = {
    "business_objective": "Help users discover connections and grow their professional/social network",
    "platform": "LinkedIn-style professional social network",
    "friendship_symmetry": "Symmetric -- both sides must accept the connection",
    "scale": {
        "total_users": "~1 billion",
        "daily_active_users": "~300 million",
        "avg_connections_per_user": "~1,000",
    },
    "graph_dynamics": "Relatively static -- social graphs don't change drastically day to day",
    "data_available": [
        "User profiles (education, work, skills, demographics)",
        "Connection graph (who is connected to whom)",
        "Interaction logs (profile views, searches, messages)",
    ],
    "key_insight": "92% of new friendships form via friends-of-friends (Meta research)"
}

print("=== PYMK System Requirements ===")
print(f"\nBusiness Objective: {requirements['business_objective']}")
print(f"Scale: {requirements['scale']['total_users']} users, {requirements['scale']['daily_active_users']} DAU")
print(f"Avg connections per user: {requirements['scale']['avg_connections_per_user']}")
print(f"Friendship symmetry: {requirements['friendship_symmetry']}")
print(f"Graph dynamics: {requirements['graph_dynamics']}")
print(f"\nKey Insight: {requirements['key_insight']}")
print(f"\nThis means FoF is our #1 candidate generation strategy!")

### Why PYMK Is a Graph Problem

| Aspect | Movie Recommendation | People You May Know |
|--------|---------------------|--------------------|
| Core data structure | User-item matrix | **Social graph** |
| Relationship type | One-directional (user rates movie) | **Bidirectional** (mutual friendship) |
| Best signal | Past ratings / watch history | **Mutual connections** (friends of friends) |
| Cold start | New movies, new users | New users (but graph structure helps!) |
| Scale challenge | Millions of items | **Billions of nodes, trillions of edges** |
| Key algorithm | Collaborative filtering | **Graph neural networks, edge prediction** |

## 2. Building a Social Graph

Before we can recommend friends, we need to represent the social network as a **graph**. Think of it like a map where:
- Each person is a **dot** (node)
- Each friendship is a **line** (edge) connecting two dots

Let's build one from scratch.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import random

random.seed(42)
np.random.seed(42)

# === Build a synthetic social network ===
# We'll create 50 users with realistic clustering
# (people tend to form friend groups, just like in school)

def build_social_network(n_users=50, n_clusters=4, intra_prob=0.4, inter_prob=0.02):
    """
    Build a synthetic social network with community structure.
    
    12-year-old version:
    Imagine 4 friend groups at school (the athletes, the gamers,
    the band kids, the theater kids). Within each group, kids are
    likely to be friends (40% chance). Between groups, it's much
    rarer (2% chance). This is exactly how real social networks work!
    
    Staff-level detail:
    This is a stochastic block model -- the standard generative model
    for networks with community structure. The parameters (intra_prob,
    inter_prob) control the modularity of the graph.
    """
    G = nx.Graph()
    
    # Assign users to clusters
    cluster_sizes = [n_users // n_clusters] * n_clusters
    cluster_sizes[-1] += n_users - sum(cluster_sizes)  # handle remainder
    
    cluster_labels = {}
    user_id = 0
    cluster_names = ['Engineering', 'Marketing', 'Research', 'Design']
    
    for cluster_idx, size in enumerate(cluster_sizes):
        for _ in range(size):
            G.add_node(user_id, cluster=cluster_idx, 
                      cluster_name=cluster_names[cluster_idx])
            cluster_labels[user_id] = cluster_idx
            user_id += 1
    
    # Add edges based on cluster membership
    for i in range(n_users):
        for j in range(i + 1, n_users):
            if cluster_labels[i] == cluster_labels[j]:
                if random.random() < intra_prob:
                    G.add_edge(i, j)
            else:
                if random.random() < inter_prob:
                    G.add_edge(i, j)
    
    return G, cluster_labels


G, cluster_labels = build_social_network()

print(f"=== Social Network Statistics ===")
print(f"Users (nodes): {G.number_of_nodes()}")
print(f"Connections (edges): {G.number_of_edges()}")
print(f"Average connections per user: {2 * G.number_of_edges() / G.number_of_nodes():.1f}")
print(f"Network density: {nx.density(G):.4f}")
print(f"\nClusters: Engineering, Marketing, Research, Design")

# Visualize
fig, ax = plt.subplots(1, 1, figsize=(10, 8))
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']
node_colors = [colors[cluster_labels[n]] for n in G.nodes()]
pos = nx.spring_layout(G, seed=42, k=0.5)

nx.draw(G, pos, ax=ax, node_color=node_colors, node_size=150,
        edge_color='#cccccc', width=0.5, alpha=0.9)

# Legend
for i, name in enumerate(['Engineering', 'Marketing', 'Research', 'Design']):
    ax.scatter([], [], c=colors[i], s=100, label=name)
ax.legend(title='Department', loc='upper left', fontsize=10)
ax.set_title('Social Network with Community Structure\n(Each dot = user, each line = friendship)', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Friends-of-Friends (FoF) -- The Baseline

The simplest and most powerful insight in PYMK: **if your friend knows someone, you might want to know them too.**

Think of it this way: your friend Alex knows a kid named Jordan. You have never met Jordan, but since Alex likes both of you, there is a decent chance you and Jordan would get along too. This is called **triadic closure** in graph theory -- triangles in social networks tend to close.

### Why FoF Is So Powerful
- According to Meta research, **92% of new friendships** on Facebook form between friends-of-friends
- It reduces the search space from 1 billion users to about 1 million (1000 friends x 1000 FoF each)
- It is computationally cheap -- just a 2-hop traversal

In [ ]:
def get_friends_of_friends(G, user_id):
    """
    Find all friends-of-friends for a given user.
    
    12-year-old version:
    1. Look at all YOUR friends
    2. For each friend, look at THEIR friends
    3. Remove people you already know (and yourself)
    4. Count how many times each person appeared -- more appearances = more mutual friends!
    
    Staff-level detail:
    This is a 2-hop BFS on the graph. The count of shared neighbors
    is the simplest graph-based feature and often the strongest predictor.
    Time complexity: O(d^2) where d is average degree.
    For avg degree 1000: 1000 * 1000 = 1M candidates per user.
    """
    friends = set(G.neighbors(user_id))
    fof_counts = {}  # candidate -> number of mutual friends
    
    for friend in friends:
        for fof in G.neighbors(friend):
            # Skip if fof is the user themselves or already a friend
            if fof == user_id or fof in friends:
                continue
            fof_counts[fof] = fof_counts.get(fof, 0) + 1
    
    # Sort by number of mutual friends (descending)
    ranked_fof = sorted(fof_counts.items(), key=lambda x: -x[1])
    return ranked_fof


# Demo: Find FoF for user 0
target_user = 0
friends = list(G.neighbors(target_user))
fof_results = get_friends_of_friends(G, target_user)

print(f"=== FoF Analysis for User {target_user} ===")
print(f"User {target_user}'s cluster: {G.nodes[target_user]['cluster_name']}")
print(f"User {target_user}'s direct friends: {len(friends)}")
print(f"FoF candidates found: {len(fof_results)}")

print(f"\nTop 10 FoF Candidates (ranked by mutual friends):")
print(f"{'Candidate':>10} {'Mutual Friends':>15} {'Cluster':>15}")
print("-" * 42)
for candidate, mutual_count in fof_results[:10]:
    cluster = G.nodes[candidate]['cluster_name']
    print(f"{candidate:>10} {mutual_count:>15} {cluster:>15}")

print(f"\nNotice: most top candidates are in the SAME cluster (department).")
print(f"This makes sense -- your colleagues' colleagues are likely your colleagues too!")

In [ ]:
# Visualize FoF for user 0
target_user = 0
friends = set(G.neighbors(target_user))
fof_results = get_friends_of_friends(G, target_user)
top_fof = set([c for c, _ in fof_results[:5]])

fig, ax = plt.subplots(1, 1, figsize=(10, 8))

# Color nodes by relationship to target
node_colors_fof = []
node_sizes = []
for n in G.nodes():
    if n == target_user:
        node_colors_fof.append('#FF0000')  # Red = target user
        node_sizes.append(400)
    elif n in friends:
        node_colors_fof.append('#4ECDC4')  # Teal = direct friends
        node_sizes.append(200)
    elif n in top_fof:
        node_colors_fof.append('#FFD700')  # Gold = top FoF candidates
        node_sizes.append(300)
    else:
        node_colors_fof.append('#DDDDDD')  # Gray = everyone else
        node_sizes.append(80)

nx.draw(G, pos, ax=ax, node_color=node_colors_fof, node_size=node_sizes,
        edge_color='#cccccc', width=0.5, alpha=0.9)

# Add labels for target and top candidates
labels = {target_user: f'User {target_user}'}
for c, count in fof_results[:5]:
    labels[c] = f'{c} ({count} mutual)'
nx.draw_networkx_labels(G, pos, labels, font_size=8, font_weight='bold', ax=ax)

ax.scatter([], [], c='#FF0000', s=150, label='Target User')
ax.scatter([], [], c='#4ECDC4', s=100, label='Direct Friends')
ax.scatter([], [], c='#FFD700', s=100, label='Top FoF Suggestions')
ax.scatter([], [], c='#DDDDDD', s=60, label='Other Users')
ax.legend(loc='upper left', fontsize=10)
ax.set_title('Friends-of-Friends Visualization\n(Gold nodes = top PYMK suggestions)', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Graph-Based Features

FoF gives us candidates, but we need to **rank** them. Which FoF candidate is most likely to become an actual friend? We compute **graph-based features** that capture different aspects of the relationship.

### The Three Key Graph Features

Think of these like different ways to measure how "close" two people are in the friend network:

1. **Common Neighbors (CN):** How many mutual friends do they share? (Simple counting)
2. **Jaccard Similarity:** What *fraction* of their combined friends are mutual? (Normalized counting)
3. **Adamic-Adar Index:** Mutual friends who themselves have few friends count MORE. (Weighted counting)

Why does Adamic-Adar matter? Imagine two mutual friends:
- Friend A has 5 total friends (a very selective person)
- Friend B has 5,000 friends (adds everyone)

Friend A knowing both of you is a MUCH stronger signal than Friend B knowing both of you. Adamic-Adar captures this.

In [ ]:
import math

def compute_graph_features(G, user_a, user_b):
    """
    Compute graph-based features for a pair of users.
    
    12-year-old version:
    - Common Neighbors: 'How many friends do we BOTH have?'
    - Jaccard: 'Out of ALL our friends combined, what fraction do we share?'
    - Adamic-Adar: 'Our picky mutual friends count more than our social butterfly friends.'
    
    Staff-level detail:
    These are classic link prediction heuristics from network science.
    CN is O(d), Jaccard is O(d), Adamic-Adar is O(d) where d = degree.
    In practice, CN alone is the strongest single feature.
    """
    neighbors_a = set(G.neighbors(user_a))
    neighbors_b = set(G.neighbors(user_b))
    
    # 1. Common Neighbors
    common = neighbors_a & neighbors_b
    cn = len(common)
    
    # 2. Jaccard Similarity
    union = neighbors_a | neighbors_b
    jaccard = cn / len(union) if len(union) > 0 else 0.0
    
    # 3. Adamic-Adar Index
    # Sum of 1/log(degree) for each common neighbor
    adamic_adar = 0.0
    for neighbor in common:
        degree = G.degree(neighbor)
        if degree > 1:
            adamic_adar += 1.0 / math.log(degree)
    
    # 4. Preferential Attachment
    # Product of degrees -- high-degree nodes are more likely to form new connections
    pref_attach = G.degree(user_a) * G.degree(user_b)
    
    return {
        'common_neighbors': cn,
        'jaccard_similarity': round(jaccard, 4),
        'adamic_adar': round(adamic_adar, 4),
        'preferential_attachment': pref_attach,
        'degree_a': G.degree(user_a),
        'degree_b': G.degree(user_b),
    }


# Compute features for top FoF candidates of user 0
target_user = 0
fof_results = get_friends_of_friends(G, target_user)

print(f"=== Graph Features for User {target_user}'s Top FoF Candidates ===")
print(f"{'Candidate':>10} {'CN':>5} {'Jaccard':>8} {'Adamic-Adar':>12} {'Pref Attach':>12} {'Cluster':>12}")
print("-" * 62)

for candidate, _ in fof_results[:10]:
    features = compute_graph_features(G, target_user, candidate)
    cluster = G.nodes[candidate]['cluster_name']
    print(f"{candidate:>10} {features['common_neighbors']:>5} {features['jaccard_similarity']:>8.4f} "
          f"{features['adamic_adar']:>12.4f} {features['preferential_attachment']:>12} {cluster:>12}")

In [ ]:
# Visualize how the three features compare

# Compute features for ALL FoF candidates
candidates_data = []
for candidate, mutual_count in fof_results:
    features = compute_graph_features(G, target_user, candidate)
    features['candidate'] = candidate
    features['same_cluster'] = int(G.nodes[candidate]['cluster'] == G.nodes[target_user]['cluster'])
    candidates_data.append(features)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

cn_vals = [d['common_neighbors'] for d in candidates_data]
jaccard_vals = [d['jaccard_similarity'] for d in candidates_data]
aa_vals = [d['adamic_adar'] for d in candidates_data]
same_cluster = [d['same_cluster'] for d in candidates_data]
colors_scatter = ['#FF6B6B' if s else '#45B7D1' for s in same_cluster]

axes[0].scatter(cn_vals, range(len(cn_vals)), c=colors_scatter, alpha=0.7, s=50)
axes[0].set_xlabel('Common Neighbors', fontsize=12)
axes[0].set_ylabel('Candidate Index', fontsize=12)
axes[0].set_title('Common Neighbors\n(Raw count of mutual friends)', fontsize=11)

axes[1].scatter(jaccard_vals, range(len(jaccard_vals)), c=colors_scatter, alpha=0.7, s=50)
axes[1].set_xlabel('Jaccard Similarity', fontsize=12)
axes[1].set_ylabel('Candidate Index', fontsize=12)
axes[1].set_title('Jaccard Similarity\n(Fraction of shared friends)', fontsize=11)

axes[2].scatter(aa_vals, range(len(aa_vals)), c=colors_scatter, alpha=0.7, s=50)
axes[2].set_xlabel('Adamic-Adar Index', fontsize=12)
axes[2].set_ylabel('Candidate Index', fontsize=12)
axes[2].set_title('Adamic-Adar Index\n(Picky mutual friends count more)', fontsize=11)

# Add legend to first plot
axes[0].scatter([], [], c='#FF6B6B', s=50, label='Same cluster')
axes[0].scatter([], [], c='#45B7D1', s=50, label='Different cluster')
axes[0].legend(fontsize=9)

plt.suptitle(f'Graph Feature Comparison for User {target_user}\'s FoF Candidates', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Red dots = candidates in the SAME department/cluster")
print("Blue dots = candidates in a DIFFERENT department/cluster")
print("Notice: same-cluster candidates tend to have higher feature values across all three metrics.")

## 5. Framing as an ML Task

### Two Approaches: Pointwise vs Edge Prediction

#### Approach 1: Pointwise Learning to Rank (LTR)

Take two users' profiles, feed them into a classifier, and predict: "Will they become friends?"

- **Pros:** Simple, any classifier works (logistic regression, XGBoost, neural net)
- **Cons:** Ignores the graph context entirely! It cannot see that User A and User B have 10 mutual friends.

#### Approach 2: Edge Prediction (Graph-Based) -- THE RECOMMENDED APPROACH

Instead of looking at two profiles in isolation, look at the **entire social graph**. Use graph features (common neighbors, Jaccard, Adamic-Adar) and Graph Neural Networks to predict missing edges.

- **Pros:** Captures social context, mutual connections, neighborhood structure
- **Cons:** More complex, needs graph infrastructure

**Why edge prediction wins:** Consider two scenarios:
- **Scenario 1:** User A and User B share 10 mutual connections -- high chance of connecting
- **Scenario 2:** User A and User B have separate friend groups with no overlap -- low chance

Pointwise LTR literally cannot distinguish these scenarios. Edge prediction can.

In [ ]:
# === Demonstrate why graph context matters ===

# Find two pairs with similar profile features but DIFFERENT graph features
# Pair 1: two users in the same cluster with many mutual friends
# Pair 2: two users in different clusters with no mutual friends

def find_contrast_pairs(G, cluster_labels):
    """Find pairs that illustrate why graph features matter."""
    # Find a pair with high common neighbors (same cluster, not connected)
    best_high = None
    best_high_cn = 0
    
    # Find a pair with zero common neighbors (different cluster, not connected)
    best_low = None
    
    for i in range(G.number_of_nodes()):
        for j in range(i + 1, G.number_of_nodes()):
            if G.has_edge(i, j):
                continue  # Skip already-connected pairs
            
            cn = len(set(G.neighbors(i)) & set(G.neighbors(j)))
            
            if cn > best_high_cn and cluster_labels[i] == cluster_labels[j]:
                best_high_cn = cn
                best_high = (i, j, cn)
            
            if cn == 0 and cluster_labels[i] != cluster_labels[j] and best_low is None:
                best_low = (i, j, cn)
    
    return best_high, best_low


pair_high, pair_low = find_contrast_pairs(G, cluster_labels)

print("=== Why Graph Context Matters ===")
print(f"\nPair 1 (HIGH likelihood of connecting):")
print(f"  Users: {pair_high[0]} and {pair_high[1]}")
print(f"  Same cluster: YES ({G.nodes[pair_high[0]]['cluster_name']})")
print(f"  Common neighbors: {pair_high[2]}")
print(f"  --> These two share MANY mutual friends. Very likely to connect!")

print(f"\nPair 2 (LOW likelihood of connecting):")
print(f"  Users: {pair_low[0]} and {pair_low[1]}")
print(f"  Same cluster: NO ({G.nodes[pair_low[0]]['cluster_name']} vs {G.nodes[pair_low[1]]['cluster_name']})")
print(f"  Common neighbors: {pair_low[2]}")
print(f"  --> These two have ZERO mutual friends. Unlikely to connect.")

print(f"\n  A pointwise model looking only at profiles might give similar scores.")
print(f"  A graph-based model sees the massive difference in social context.")

## 6. Training Data Construction

To train our model, we need labeled examples: pairs of users who DID connect (positive) and pairs who did NOT (negative).

### The Temporal Split Trick

We use a **time-based split** to avoid data leakage:
1. Take the graph at time **t** (this is the model's input)
2. Look at the graph at time **t+1**
3. New edges that appeared between t and t+1 are **positive labels**
4. FoF pairs that did NOT form edges are **negative labels**

This way, the model learns to predict the future from the past -- exactly what we need in production.

In [ ]:
def construct_training_data(G, positive_ratio=0.15):
    """
    Construct training data from the social graph.
    
    12-year-old version:
    We take a snapshot of the friend network today. Then we look at
    who became friends next week. The new friendships are our 'yes' examples.
    The people who COULD have become friends (FoF) but didn't are 'no' examples.
    
    Staff-level detail:
    We simulate a temporal split by removing some edges (future connections)
    and using the remaining graph as the observed state. Negative sampling
    is done from FoF pairs that don't have edges -- this is more realistic
    than random negative sampling because these are the actual candidates
    the system would consider.
    """
    edges = list(G.edges())
    n_positive = int(len(edges) * positive_ratio)
    
    # Simulate: remove some edges to create "future" connections (positive examples)
    random.shuffle(edges)
    positive_edges = edges[:n_positive]
    
    # Create observed graph (without positive edges)
    G_observed = G.copy()
    G_observed.remove_edges_from(positive_edges)
    
    # Positive examples: edges that will form
    positive_samples = []
    for u, v in positive_edges:
        features = compute_graph_features(G_observed, u, v)
        features['user_a'] = u
        features['user_b'] = v
        features['same_cluster'] = int(cluster_labels[u] == cluster_labels[v])
        features['label'] = 1
        positive_samples.append(features)
    
    # Negative examples: FoF pairs that did NOT form edges
    negative_samples = []
    all_nodes = list(G_observed.nodes())
    attempts = 0
    while len(negative_samples) < n_positive * 3 and attempts < n_positive * 30:
        u = random.choice(all_nodes)
        friends_u = set(G_observed.neighbors(u))
        if not friends_u:
            attempts += 1
            continue
        # Pick a FoF
        friend = random.choice(list(friends_u))
        fof_candidates = set(G_observed.neighbors(friend)) - friends_u - {u}
        if not fof_candidates:
            attempts += 1
            continue
        v = random.choice(list(fof_candidates))
        
        # Make sure this pair is not in the positive set
        if (u, v) in positive_edges or (v, u) in positive_edges:
            attempts += 1
            continue
        if G_observed.has_edge(u, v):
            attempts += 1
            continue
        
        features = compute_graph_features(G_observed, u, v)
        features['user_a'] = u
        features['user_b'] = v
        features['same_cluster'] = int(cluster_labels[u] == cluster_labels[v])
        features['label'] = 0
        negative_samples.append(features)
        attempts += 1
    
    return positive_samples + negative_samples, G_observed


training_samples, G_observed = construct_training_data(G)

print(f"=== Training Data ===")
n_pos = sum(1 for s in training_samples if s['label'] == 1)
n_neg = sum(1 for s in training_samples if s['label'] == 0)
print(f"Total samples: {len(training_samples)}")
print(f"Positive (will connect): {n_pos} ({n_pos/len(training_samples)*100:.1f}%)")
print(f"Negative (will NOT connect): {n_neg} ({n_neg/len(training_samples)*100:.1f}%)")
print(f"\nNote: In real systems, the imbalance is much worse (~1:100 or more).")
print(f"Here we use ~1:3 for demonstration purposes.")

print(f"\nSample positive example:")
pos_example = [s for s in training_samples if s['label'] == 1][0]
for k, v in pos_example.items():
    print(f"  {k}: {v}")

## 7. Training a Classifier

Now we train a model to predict whether two users will connect, given their graph features. We start with a simple classifier (Gradient Boosted Trees) and compare with a neural network.

In production, this would be a GNN (covered in Notebook 02), but the feature-based classifier is an excellent baseline that is faster to train and easier to debug.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report, average_precision_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Prepare feature matrix
feature_cols = ['common_neighbors', 'jaccard_similarity', 'adamic_adar',
                'preferential_attachment', 'degree_a', 'degree_b', 'same_cluster']

X = np.array([[s[f] for f in feature_cols] for s in training_samples])
y = np.array([s['label'] for s in training_samples])

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale features for logistic regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- Model 1: Logistic Regression (baseline) ---
lr_model = LogisticRegression(max_iter=1000, class_weight='balanced')
lr_model.fit(X_train_scaled, y_train)
lr_probs = lr_model.predict_proba(X_test_scaled)[:, 1]
lr_auc = roc_auc_score(y_test, lr_probs)
lr_ap = average_precision_score(y_test, lr_probs)

# --- Model 2: Gradient Boosted Trees ---
gbdt_model = GradientBoostingClassifier(
    n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42
)
gbdt_model.fit(X_train, y_train)  # No scaling needed for trees
gbdt_probs = gbdt_model.predict_proba(X_test)[:, 1]
gbdt_auc = roc_auc_score(y_test, gbdt_probs)
gbdt_ap = average_precision_score(y_test, gbdt_probs)

print("=== Model Comparison ===")
print(f"{'Model':<25} {'ROC-AUC':>10} {'Avg Precision':>15}")
print("-" * 52)
print(f"{'Logistic Regression':<25} {lr_auc:>10.4f} {lr_ap:>15.4f}")
print(f"{'Gradient Boosted Trees':<25} {gbdt_auc:>10.4f} {gbdt_ap:>15.4f}")

# Feature importance
print(f"\n=== Feature Importance (GBDT) ===")
importances = gbdt_model.feature_importances_
sorted_idx = np.argsort(importances)[::-1]
for i, idx in enumerate(sorted_idx):
    bar = '#' * int(importances[idx] * 50)
    print(f"  {feature_cols[idx]:<25} {importances[idx]:.4f} {bar}")

print(f"\nKey takeaway: common_neighbors (mutual friends) is typically the strongest feature.")
print(f"This aligns with Meta's finding that 92% of connections come from FoF!")

In [ ]:
# Visualize model performance
from sklearn.metrics import roc_curve, precision_recall_curve

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
for name, probs, color in [('Logistic Regression', lr_probs, '#FF6B6B'),
                             ('GBDT', gbdt_probs, '#4ECDC4')]:
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc = roc_auc_score(y_test, probs)
    axes[0].plot(fpr, tpr, color=color, linewidth=2, label=f'{name} (AUC={auc:.3f})')

axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random (AUC=0.500)')
axes[0].set_xlabel('False Positive Rate', fontsize=12)
axes[0].set_ylabel('True Positive Rate', fontsize=12)
axes[0].set_title('ROC Curve', fontsize=14)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Precision-Recall Curve
for name, probs, color in [('Logistic Regression', lr_probs, '#FF6B6B'),
                             ('GBDT', gbdt_probs, '#4ECDC4')]:
    precision, recall, _ = precision_recall_curve(y_test, probs)
    ap = average_precision_score(y_test, probs)
    axes[1].plot(recall, precision, color=color, linewidth=2, label=f'{name} (AP={ap:.3f})')

axes[1].set_xlabel('Recall', fontsize=12)
axes[1].set_ylabel('Precision', fontsize=12)
axes[1].set_title('Precision-Recall Curve', fontsize=14)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.suptitle('PYMK Classifier Performance', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print("ROC-AUC measures how well we separate 'will connect' from 'will not connect'.")
print("Precision-Recall is more informative when classes are imbalanced (which they are in PYMK).")

## 8. User-Level Features

Graph features are powerful, but we can make our model even better by adding **user profile features** and **user-user affinity features**.

Think of these as the information on a person's trading card:

### User Features (per person)
- **Demographics:** age, gender, city, country
- **Connection stats:** number of connections, followers, pending requests
- **Account age:** newer accounts might be spam; older ones are more trustworthy
- **Activity level:** likes received, posts made, profile completeness

### User-User Affinity Features (per pair)
- **Schools in common:** did they attend the same school?
- **Contemporaries at school:** did they overlap in years? (Much more predictive!)
- **Companies in common:** worked at the same place?
- **Same industry:** both in tech? both in finance?
- **Profile visits:** has User A viewed User B's profile?

In [ ]:
# === Enrich nodes with user profile features ===

companies = ['Google', 'Meta', 'Amazon', 'Apple', 'Netflix', 'Microsoft', 'Stripe', 'Airbnb']
schools = ['MIT', 'Stanford', 'CMU', 'Berkeley', 'Harvard', 'Georgia Tech', 'Waterloo']
industries = ['Tech', 'Finance', 'Healthcare', 'Education', 'Media']
cities = ['San Francisco', 'New York', 'Seattle', 'Austin', 'Boston', 'Chicago']

def add_user_features(G, cluster_labels):
    """
    Add realistic user profile features to each node.
    Users in the same cluster tend to share company/school.
    """
    for node in G.nodes():
        cluster = cluster_labels[node]
        
        # Users in same cluster are more likely to share attributes
        # (simulating real-world clustering by workplace/school)
        G.nodes[node]['company'] = companies[cluster] if random.random() < 0.6 else random.choice(companies)
        G.nodes[node]['school'] = schools[cluster] if random.random() < 0.5 else random.choice(schools)
        G.nodes[node]['industry'] = industries[cluster % len(industries)] if random.random() < 0.7 else random.choice(industries)
        G.nodes[node]['city'] = cities[cluster % len(cities)] if random.random() < 0.5 else random.choice(cities)
        G.nodes[node]['age'] = random.randint(22, 55)
        G.nodes[node]['connections'] = G.degree(node)
        G.nodes[node]['account_age_days'] = random.randint(30, 3650)
        G.nodes[node]['profile_completeness'] = round(random.uniform(0.3, 1.0), 2)
    
    return G


def compute_affinity_features(G, user_a, user_b):
    """
    Compute user-user affinity features.
    
    12-year-old version:
    'Did these two kids go to the same school? Same chess club? Same neighborhood?'
    
    Staff-level detail:
    These features capture attribute similarity independent of graph structure.
    Combined with graph features, they give a much richer signal.
    """
    a = G.nodes[user_a]
    b = G.nodes[user_b]
    
    return {
        'same_company': int(a['company'] == b['company']),
        'same_school': int(a['school'] == b['school']),
        'same_industry': int(a['industry'] == b['industry']),
        'same_city': int(a['city'] == b['city']),
        'age_diff': abs(a['age'] - b['age']),
        'avg_profile_completeness': (a['profile_completeness'] + b['profile_completeness']) / 2,
    }


G = add_user_features(G, cluster_labels)

# Show features for a few users
print("=== User Profile Features (Sample) ===")
print(f"{'User':>5} {'Company':>12} {'School':>10} {'Industry':>12} {'City':>15} {'Age':>5} {'Connections':>12}")
print("-" * 75)
for node in range(5):
    n = G.nodes[node]
    print(f"{node:>5} {n['company']:>12} {n['school']:>10} {n['industry']:>12} {n['city']:>15} {n['age']:>5} {n['connections']:>12}")

# Show affinity features for a pair
print(f"\n=== Affinity Features: User 0 vs User 1 ===")
affinity = compute_affinity_features(G, 0, 1)
for k, v in affinity.items():
    print(f"  {k}: {v}")

## 9. Time-Discounted Mutual Connections

This is a clever feature that impresses interviewers. The idea:

Not all mutual connections are equally important. A mutual friend you made **last week** is worth more than one you made **5 years ago**. Why?

- **Scenario 1:** User A recently connected with C, D, E (network is growing) -- User A is actively expanding their network and likely to add more people
- **Scenario 2:** User A connected with C, D, E five years ago (stable network) -- User A probably already knows about User B and chose NOT to connect

We apply **exponential time decay** to weight recent connections more heavily.

In [ ]:
import math

def time_discounted_mutual_connections(G, user_a, user_b, current_time, decay_rate=0.1):
    """
    Compute time-discounted mutual connection score.
    
    12-year-old version:
    'New mutual friends count more than old ones. If you JUST became
    friends with someone who also knows Jordan, that is a stronger signal
    that you should meet Jordan than if you have known that mutual friend
    for 10 years.'
    
    Staff-level detail:
    weight(mutual_connection) = exp(-decay_rate * days_since_connection)
    This exponential decay gives a smooth decrease over time.
    decay_rate controls the half-life: 0.1 means the weight halves
    every ~7 days.
    """
    neighbors_a = set(G.neighbors(user_a))
    neighbors_b = set(G.neighbors(user_b))
    mutual = neighbors_a & neighbors_b
    
    if not mutual:
        return 0.0, 0
    
    total_weight = 0.0
    for m in mutual:
        # Simulate connection timestamps (days ago)
        days_ago = random.randint(1, 365)
        weight = math.exp(-decay_rate * days_ago)
        total_weight += weight
    
    return total_weight, len(mutual)


# Demonstrate time decay
days_range = np.arange(0, 365, 1)
decay_rates = [0.01, 0.05, 0.1, 0.2]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Decay curves for different rates
for rate in decay_rates:
    weights = np.exp(-rate * days_range)
    axes[0].plot(days_range, weights, linewidth=2, label=f'decay_rate={rate}')

axes[0].set_xlabel('Days Since Connection', fontsize=12)
axes[0].set_ylabel('Weight', fontsize=12)
axes[0].set_title('Time Decay Functions\n(Higher decay = recent connections matter more)', fontsize=12)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Plot 2: Compare raw vs time-discounted for a specific pair
scenarios = {
    'All connections\nmade this week': [1, 2, 3, 4, 5],
    'All connections\nmade 6 months ago': [150, 160, 170, 180, 190],
    'Mix of\nrecent and old': [2, 30, 90, 180, 300],
}

raw_counts = []
discounted_scores = []
labels = []

for label, days_list in scenarios.items():
    raw = len(days_list)
    discounted = sum(math.exp(-0.05 * d) for d in days_list)
    raw_counts.append(raw)
    discounted_scores.append(discounted)
    labels.append(label)

x_pos = np.arange(len(labels))
width = 0.35
axes[1].bar(x_pos - width/2, raw_counts, width, label='Raw Count', color='#FF6B6B', alpha=0.8)
axes[1].bar(x_pos + width/2, discounted_scores, width, label='Time-Discounted', color='#4ECDC4', alpha=0.8)
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(labels, fontsize=10)
axes[1].set_ylabel('Score', fontsize=12)
axes[1].set_title('Raw Count vs Time-Discounted Score\n(All have 5 mutual friends)', fontsize=12)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("Key insight: All three scenarios have 5 mutual friends (same raw count).")
print("But the time-discounted score correctly distinguishes:")
print("  - Recent connections = high score (user is actively networking)")
print("  - Old connections = low score (user chose NOT to connect long ago)")

## 10. Evaluation Metrics

### Offline Metrics

| Metric | What It Measures | Why We Use It |
|--------|-----------------|---------------|
| **ROC-AUC** | How well we separate "will connect" from "will not" | Standard for binary classification |
| **mAP** (mean Average Precision) | How well we rank suggestions (relevant ones at top) | Captures ranking quality with binary labels |

### Online Metrics

| Metric | What It Measures | Drawback |
|--------|-----------------|----------|
| Connection requests sent | Are people clicking "Connect"? | Users might spam requests |
| **Connection requests accepted** | Are real connections forming? | **This is the north star metric** |

Why "requests accepted" is the north star: a connection only forms when BOTH sides agree. This measures actual network growth, which is the business goal.

In [ ]:
def compute_map_at_k(y_true_per_user, y_scores_per_user, k=10):
    """
    Compute mean Average Precision at k.
    
    12-year-old version:
    For each user, we make a ranked list of suggestions. If the people
    they actually connect with appear near the TOP of the list, we get
    a high score. If they are buried at the bottom, our score is low.
    We average this across all users.
    
    Staff-level detail:
    AP@k = (1/min(m,k)) * sum_{i=1}^{k} P(i) * rel(i)
    where P(i) = precision at position i, rel(i) = 1 if item i is relevant.
    mAP = mean of AP across all queries (users).
    """
    all_aps = []
    
    for y_true, y_scores in zip(y_true_per_user, y_scores_per_user):
        # Sort by predicted scores (descending)
        sorted_indices = np.argsort(-np.array(y_scores))
        y_true_sorted = np.array(y_true)[sorted_indices][:k]
        
        # Compute AP
        n_relevant = 0
        precision_sum = 0.0
        
        for i, is_relevant in enumerate(y_true_sorted):
            if is_relevant == 1:
                n_relevant += 1
                precision_sum += n_relevant / (i + 1)
        
        total_relevant = sum(y_true)
        ap = precision_sum / min(total_relevant, k) if total_relevant > 0 else 0.0
        all_aps.append(ap)
    
    return np.mean(all_aps)


# Simulate per-user evaluation
np.random.seed(42)
n_users_eval = 100
n_suggestions = 20

y_true_per_user = []
y_scores_per_user = []

for _ in range(n_users_eval):
    # Each user sees 20 suggestions, ~3 of which they actually connect with
    y_true = [1 if random.random() < 0.15 else 0 for _ in range(n_suggestions)]
    # Good model: higher scores for relevant items (with some noise)
    y_scores = [0.8 + random.gauss(0, 0.1) if y == 1 else 0.3 + random.gauss(0, 0.15) for y in y_true]
    y_true_per_user.append(y_true)
    y_scores_per_user.append(y_scores)

# Compute mAP at different k values
k_values = [1, 3, 5, 10, 20]
map_values = [compute_map_at_k(y_true_per_user, y_scores_per_user, k=k) for k in k_values]

print("=== Offline Evaluation: mAP@k ===")
print(f"{'k':>5} {'mAP@k':>10}")
print("-" * 17)
for k, map_val in zip(k_values, map_values):
    print(f"{k:>5} {map_val:>10.4f}")

# Visualize
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
ax.plot(k_values, map_values, 'o-', color='#4ECDC4', linewidth=2, markersize=8)
ax.set_xlabel('k (number of suggestions shown)', fontsize=12)
ax.set_ylabel('mAP@k', fontsize=12)
ax.set_title('Mean Average Precision at Different k Values', fontsize=14)
ax.grid(True, alpha=0.3)
ax.set_xticks(k_values)
plt.tight_layout()
plt.show()

print("\nmAP decreases as k increases because it is harder to maintain")
print("high precision when showing more suggestions.")

## 11. Serving Architecture

### The Efficiency Challenge

With 1 billion users, comparing every pair would be 10^18 comparisons -- completely impractical! We need a smart architecture.

### Two-Pipeline Architecture

```
PIPELINE 1: PYMK GENERATION (Offline/Batch)

User --> FoF Service --> Candidate Connections (2-hop neighbors)
                              |
                              v
                     Feature Computation (graph + profile + affinity)
                              |
                              v
                     Scoring Service (GNN or classifier)
                              |
                              v
                     Ranked PYMK List --> Database


PIPELINE 2: ONLINE SERVING

User Opens App --> Check Database
                        |
           +------------+------------+
           |                         |
      Pre-computed              Not Found
      PYMK exists                    |
           |                         v
           v               Trigger Generation
      Return PYMK           Pipeline (one-time)
```

### Why Batch Over Real-Time?
1. Social graphs are **stable** -- your friends list does not change every minute
2. Computing for 300M DAU in real-time is too expensive
3. Pre-computed results can be served instantly from a database

### Smart Refresh Strategies
- Re-compute every **7 days** for established users
- Re-compute **daily** for new users (their networks grow faster)
- Pre-compute **more candidates than needed**, show unseen ones each visit

In [ ]:
# === Simulate the complete PYMK pipeline ===

class PYMKPipeline:
    """
    End-to-end PYMK generation pipeline.
    
    12-year-old version:
    1. Find your friends' friends (FoF) -- these are the candidates
    2. For each candidate, compute features (mutual friends, same school, etc.)
    3. Score each candidate with our ML model
    4. Return the top-k highest scored candidates
    """
    
    def __init__(self, G, model, scaler, feature_cols):
        self.G = G
        self.model = model
        self.scaler = scaler
        self.feature_cols = feature_cols
    
    def generate_pymk(self, user_id, top_k=10):
        """Generate PYMK suggestions for a user."""
        # Step 1: Get FoF candidates
        fof_results = get_friends_of_friends(self.G, user_id)
        
        if not fof_results:
            return []  # No FoF candidates (cold start!)
        
        # Step 2: Compute features for each candidate
        candidates = []
        feature_matrix = []
        
        for candidate_id, _ in fof_results:
            # Graph features
            graph_feats = compute_graph_features(self.G, user_id, candidate_id)
            # Affinity features
            affinity_feats = compute_affinity_features(self.G, user_id, candidate_id)
            
            # Combine into feature vector
            feature_vec = [
                graph_feats['common_neighbors'],
                graph_feats['jaccard_similarity'],
                graph_feats['adamic_adar'],
                graph_feats['preferential_attachment'],
                graph_feats['degree_a'],
                graph_feats['degree_b'],
                affinity_feats['same_company'] + affinity_feats['same_school'] +
                affinity_feats['same_industry'] + affinity_feats['same_city'],
            ]
            feature_matrix.append(feature_vec)
            candidates.append(candidate_id)
        
        # Step 3: Score with model
        X_candidates = np.array(feature_matrix)
        scores = self.model.predict_proba(X_candidates)[:, 1]
        
        # Step 4: Rank and return top-k
        ranked_indices = np.argsort(-scores)[:top_k]
        
        results = []
        for idx in ranked_indices:
            cid = candidates[idx]
            results.append({
                'candidate_id': cid,
                'score': round(float(scores[idx]), 4),
                'cluster': self.G.nodes[cid]['cluster_name'],
                'company': self.G.nodes[cid]['company'],
                'school': self.G.nodes[cid]['school'],
                'common_neighbors': compute_graph_features(self.G, user_id, cid)['common_neighbors'],
            })
        
        return results


# Run the pipeline
pipeline = PYMKPipeline(G, gbdt_model, scaler, feature_cols)
target_user = 0

print(f"=== PYMK Suggestions for User {target_user} ===")
print(f"User {target_user}: {G.nodes[target_user]['cluster_name']} department, "
      f"{G.nodes[target_user]['company']}, {G.nodes[target_user]['school']}")
print()

suggestions = pipeline.generate_pymk(target_user, top_k=10)

print(f"{'Rank':>5} {'ID':>5} {'Score':>8} {'Mutual':>8} {'Cluster':>12} {'Company':>12} {'School':>10}")
print("-" * 65)
for i, s in enumerate(suggestions):
    print(f"{i+1:>5} {s['candidate_id']:>5} {s['score']:>8.4f} {s['common_neighbors']:>8} "
          f"{s['cluster']:>12} {s['company']:>12} {s['school']:>10}")

print(f"\nTotal FoF candidates: {len(get_friends_of_friends(G, target_user))}")
print(f"Shown to user: {len(suggestions)}")
print(f"\nIn production with 1B users, FoF reduces 1B candidates to ~1M,")
print(f"then the model scores and ranks the top ~50 to show.")

## 12. Key Takeaways

1. **PYMK is fundamentally a graph problem** -- the social network structure is the most important signal

2. **Friends-of-Friends is the #1 strategy** -- 92% of new connections come from FoF; it also reduces the search space from billions to millions

3. **Three key graph features** -- Common Neighbors (simple count), Jaccard (normalized), Adamic-Adar (weighted by friend selectivity)

4. **Edge prediction > Pointwise LTR** -- graph context (mutual friends, neighborhood structure) is essential and cannot be captured by pointwise approaches

5. **Time-discounted mutual connections** -- a clever feature that distinguishes active from stable networks

6. **Batch prediction preferred** -- social graphs are stable; pre-compute PYMK offline and serve from database

7. **North star metric = connections accepted** -- not just requests sent (to avoid spam)

8. **Two-pipeline architecture** -- offline generation (FoF + scoring) + online serving (database lookup with fallback)

### What's Next
- **Notebook 02:** Graph Neural Networks for PYMK -- going beyond hand-crafted features
- **Notebook 03:** Scaling and production challenges -- handling billions of nodes
- **Notebook 04:** Interview walkthrough -- putting it all together

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)